# The Server Inspector

The MCP Python SDK ships a **browser-based inspector** (`mcp dev`) to test a server's tools/resources/prompts in isolation — no client or Claude required. It's handy for a fast build → test → debug loop.

> **This tool is optional — you can skip it.** It's a developer convenience, not part of the server or client you're building. If it won't launch/connect on your machine (see the gotchas below), verify the server another way (also below) and move on — nothing later in the section depends on the inspector.

## You don't need the inspector to verify the server

Two reliable alternatives that need no extra tooling:

1. **Import-based check (works right now).** From `04-defining-tools.ipynb`, importing the server and calling `await mcp.list_tools()` confirms the tools are registered with the right schemas. Same idea works for resources (`await mcp.list_resources()`) and prompts (`await mcp.list_prompts()`) in later lessons.
2. **End-to-end via the CLI (after the next lesson).** Once the client's `list_tools`/`call_tool` are wired (**Implementing a client**), just run `python main.py` and ask the bot to read/edit a document — that exercises the full client↔server↔Claude path, which is the real thing anyway.

So the inspector is a *nice-to-have*, not a gate.

## If you do want to run it

Activate your venv, then from the project folder:

```powershell
.\.venv\Scripts\Activate.ps1
cd building-with-the-claude-api\07-mcp\cli-project-complete
mcp dev mcp_server.py
```

It prints a local URL (proxy on **6277**, UI on **6274** in current versions). Then use the UI: **Connect** → **Tools** → **List Tools** → pick a tool → fill args → **Run Tool**. The complete server also exposes **Resources** and **Prompts**.

A good smoke test — read → edit → read `deposition.md`:

| Tool | Args | Expect |
|------|------|--------|
| `read_doc_contents` | `doc_id = deposition.md` | "...testimony of Angela Smith, P.E." |
| `edit_document` | `doc_id = deposition.md`, `old_str = Smith`, `new_str = Jones` | success |
| `read_doc_contents` | `doc_id = deposition.md` | "...Angela **Jones**, P.E." |

## Gotchas (post-course changes that commonly break `mcp dev`)

**1. Node.js / `npx` required.** `mcp dev` launches the JavaScript MCP Inspector via `npx @modelcontextprotocol/inspector`. No Node → it won't start. Install Node.js, or use the alternatives above.

**2. "Connection Error — Did you add the proxy session token in Configuration?"** Newer inspector versions require a **proxy session token** so arbitrary web pages can't reach your local MCP proxy. Opening `localhost:6274` by hand skips it. Fixes, easiest first:

- **Open the tokenized URL** the terminal printed — it looks like `http://localhost:6274/?MCP_PROXY_AUTH_TOKEN=<token>`. That pre-fills the token, and **Connect** works.
- **Paste the token** from the terminal (`Session token: ...`) into the Inspector's **⚙️ Configuration → Proxy Session Token**, then **Connect**.
- **Disable auth for local dev** (simplest; local only): stop the inspector, then
  ```powershell
  $env:DANGEROUSLY_OMIT_AUTH = "true"
  mcp dev mcp_server.py
  # later: Remove-Item Env:DANGEROUSLY_OMIT_AUTH
  ```

If none of that cooperates, don't sink time into it — the inspector is optional. Use the import check / CLI and continue.

## Takeaway

The inspector's value is the tight loop: *edit server → test a tool directly → verify → debug in isolation*, without standing up a full app. When it works, it's the fastest way to poke at a server. When it doesn't (Node/token friction), the import-based check and the CLI cover the same ground.

Next: **Implementing a client** — wire `mcp_client.py`'s `list_tools` / `call_tool` so the CLI chatbot can actually use the server's tools (and you can test the whole thing by chatting).